In [1]:
import pandas as pd
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
import requests 
import os
import numpy as np
import pyarrow
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path



In [2]:
git_folder = "Patricia-Promise-Immo"
folder_entries = "data"
base_dir = Path().resolve().parents[1]
data_dir = base_dir/git_folder/folder_entries
if not data_dir.exists():
    data_dir.mkdir(parents=True)
print(f"Data directory is set to : {data_dir}")

Data directory is set to : C:\Users\promi\Projets\Personnel\Patricia-Promise-Immo\Patricia-Promise-Immo\data


In [3]:

DOWNLOADs_ = [
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234902/valeursfoncieres-2025-s1.txt.zip',
        'year' : '2025'
    },
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234857/valeursfoncieres-2024.txt.zip',
        'year' : '2024'
    },
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234851/valeursfoncieres-2023.txt.zip',
        'year' : '2023'
    },
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234844/valeursfoncieres-2022.txt.zip',
        'year' : '2022'
    },
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234836/valeursfoncieres-2021.txt.zip',
        'year' : '2021'
    },
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234831/valeursfoncieres-2020-s2.txt.zip',
        'year' : '2020'
    }
    
    ]


for immo_file in DOWNLOADs_ :
    response = requests.get(immo_file['url'], stream=True) #Stream = True afin d'éviter de tout charger en mémoire
    
    # Enregistrement du fichier par itérations
    with open(f"{data_dir}/immo_entries_{immo_file['year']}.txt.zip", 'wb') as file:
        for chunk in response.iter_content(chunk_size=10_000):
            file.write(chunk)



In [4]:
all_immo_files = os.listdir(data_dir)

In [5]:
for immo_file in all_immo_files:
    df = pd.read_csv(f"{data_dir}/{immo_file}", nrows=100, on_bad_lines='skip', low_memory=False, sep='|', decimal=',', dtype={'Code postal':str, 'Valeur fonciere':np.float64})
    print(df.columns.to_list())


['Identifiant de document', 'Reference document', '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 'No disposition', 'Date mutation', 'Nature mutation', 'Valeur fonciere', 'No voie', 'B/T/Q', 'Type de voie', 'Code voie', 'Voie', 'Code postal', 'Commune', 'Code departement', 'Code commune', 'Prefixe de section', 'Section', 'No plan', 'No Volume', '1er lot', 'Surface Carrez du 1er lot', '2eme lot', 'Surface Carrez du 2eme lot', '3eme lot', 'Surface Carrez du 3eme lot', '4eme lot', 'Surface Carrez du 4eme lot', '5eme lot', 'Surface Carrez du 5eme lot', 'Nombre de lots', 'Code type local', 'Type local', 'Identifiant local', 'Surface reelle bati', 'Nombre pieces principales', 'Nature culture', 'Nature culture speciale', 'Surface terrain']
['Identifiant de document', 'Reference document', '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 'No disposition', 'Date mutation', 'Nature mutation', 'Valeur fonciere', 'N

In [6]:
# enregistrement des csv en 1 seul par chunks pour épargner la machine

first_file = True

for immo_file in all_immo_files:
    for index, chunk in enumerate(pd.read_csv(f'{data_dir}/{immo_file}', on_bad_lines='skip', low_memory=False, sep='|', chunksize=10_000, decimal=',', dtype={'Code postal':str, 'Valeur fonciere':np.float64})):
        df.to_csv('all_immo_entries.csv', mode = 'w' if first_file else 'a', header=first_file)
        first_file = False
    print(f'"{immo_file}" : DONE')

"immo_entries_2020.txt.zip" : DONE
"immo_entries_2021.txt.zip" : DONE
"immo_entries_2022.txt.zip" : DONE
"immo_entries_2023.txt.zip" : DONE
"immo_entries_2024.txt.zip" : DONE
"immo_entries_2025.txt.zip" : DONE


In [7]:

writer = None

for index, chunk in enumerate(pd.read_csv('all_immo_entries.csv', chunksize=10_000)):
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    
    # Initialise le writer une seule fois avec le schéma du premier chunk
    if writer is None:
        writer = pq.ParquetWriter('all_immo_entries.parquet', schema=table.schema, compression="snappy")
    
    # Écrit le chunk courant
    writer.write_table(table)

# Ferme le writer à la fin
if writer is not None:
    writer.close()



In [8]:
df = pd.read_parquet('all_immo_entries.parquet')

print("Parquet file info:")
df.head(20)
df.info()


Parquet file info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201300 entries, 0 to 201299
Data columns (total 44 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Unnamed: 0                  201300 non-null  int64  
 1   Identifiant de document     0 non-null       float64
 2   Reference document          0 non-null       float64
 3   1 Articles CGI              0 non-null       float64
 4   2 Articles CGI              0 non-null       float64
 5   3 Articles CGI              0 non-null       float64
 6   4 Articles CGI              0 non-null       float64
 7   5 Articles CGI              0 non-null       float64
 8   No disposition              201300 non-null  int64  
 9   Date mutation               201300 non-null  object 
 10  Nature mutation             201300 non-null  object 
 11  Valeur fonciere             201300 non-null  float64
 12  No voie                     134871 non-null  float64


In [9]:
df.shape

(201300, 44)